In [20]:
import pandas as pd
import numpy as np
import joblib

In [21]:
df = pd.read_csv("../data/processed/healthcare_processed.csv")

raw_df = pd.read_csv("../data/raw/HealthCare.csv")

X = df.drop(columns=["No_show"])
y = df["No_show"]

groups = raw_df["PatientId"]

print("Dataset shape:", X.shape)

Dataset shape: (110527, 17)


In [22]:
#Recreate the patient-level development split

In [23]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Development data:", X_train.shape)
print("Final test data:", X_test.shape)

Development data: (88491, 17)
Final test data: (22036, 17)


In [24]:
# Define the preprocessing

In [25]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "Age",
    "Scholarship",
    "Hipertension",
    "Diabetes",
    "Alcoholism",
    "Handcap",
    "SMS_received",
    "WaitingDays",
    "ScheduledHour",
    "AppointmentMonth"
]

categorical_features = [
    "Gender",
    "Neighbourhood",
    "ScheduledDayOfWeek",
    "AppointmentDayOfWeek",
    "AgeGroup",
    "WaitingGroup",
    "ScheduledTimeGroup"
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [26]:
# Create the final Random Forest

In [27]:
from sklearn.ensemble import RandomForestClassifier

production_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        max_features="sqrt",
        min_samples_leaf=1,
        min_samples_split=2,
        random_state=42,
        n_jobs=-1
    ))
])

print("Lightweight production model created.")

Lightweight production model created.


In [28]:
# Train on all development data

In [29]:
production_model.fit(
    X_train,
    y_train
)

print("Production model training completed.")

Production model training completed.


In [30]:
# Test that the model works

In [31]:
test_probability = production_model.predict_proba(
    X_test
)[:, 1]

test_prediction = (
    test_probability >= 0.24
).astype(int)

print("Test predictions generated successfully.")
print("First 10 probabilities:", test_probability[:10])
print("First 10 predictions:", test_prediction[:10])

Test predictions generated successfully.
First 10 probabilities: [0.02502504 0.17490331 0.29256162 0.27936816 0.25751629 0.28556613
 0.25681915 0.03402176 0.36890535 0.08141802]
First 10 predictions: [0 0 1 1 1 1 1 0 1 0]


In [32]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

print("Lightweight Model Evaluation")
print("-" * 40)

print("Accuracy:",
      accuracy_score(y_test, test_prediction))

print("Precision:",
      precision_score(y_test, test_prediction))

print("Recall:",
      recall_score(y_test, test_prediction))

print("F1:",
      f1_score(y_test, test_prediction))

print("ROC-AUC:",
      roc_auc_score(y_test, test_probability))

print("PR-AUC:",
      average_precision_score(y_test, test_probability))

Lightweight Model Evaluation
----------------------------------------
Accuracy: 0.6431748048647667
Precision: 0.325739855588325
Recall: 0.7220468890892696
F1: 0.44894526596117457
ROC-AUC: 0.736297922729322
PR-AUC: 0.3675776541281196


In [33]:
joblib.dump(
    production_model,
    "../models/model.pkl",
    compress=9
)

print("Compressed production model saved successfully.")

Compressed production model saved successfully.


In [34]:
# Verify the saved model

In [35]:
loaded_model = joblib.load(
    "../models/model.pkl"
)

print("Model loaded successfully.")
print(type(loaded_model))

Model loaded successfully.
<class 'sklearn.pipeline.Pipeline'>


In [36]:
loaded_probability = loaded_model.predict_proba(
    X_test.head(5)
)[:, 1]

print("Loaded model probabilities:")
print(loaded_probability)

Loaded model probabilities:
[0.02502504 0.17490331 0.29256162 0.27936816 0.25751629]
